## Imports

In [4]:
import re
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

## Load Raw Data

In [5]:
raw_data_dir = Path("../data/raw")
languages = ["english", "tagalog", "bikolano", "ilonggo", "waray", "masbatenyo"]

# Dictionary to store raw data: {language: {(book, chapter, verse): text}}
raw_data = {}

for lang in languages:
    file_path = raw_data_dir / f"{lang}_raw.txt"
    raw_data[lang] = {}
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Parse format: BOOK CHAPTER:VERSE TEXT
            match = re.match(r'^(\w+)\s+(\d+):(\d+)\s+(.+)$', line)
            if match:
                book, chapter, verse, text = match.groups()
                key = (book, int(chapter), int(verse))
                raw_data[lang][key] = text
    
    print(f"Loaded {lang}: {len(raw_data[lang])} verses")

print(f"\nTotal languages loaded: {len(raw_data)}")

Loaded english: 3007 verses
Loaded tagalog: 3033 verses
Loaded bikolano: 3141 verses
Loaded ilonggo: 3124 verses
Loaded waray: 3134 verses
Loaded masbatenyo: 3079 verses

Total languages loaded: 6


## Data Cleaning Functions

Define functions to clean and normalize the text data.

In [6]:
def clean_text(text):
    """
    Clean and normalize text:
    - Remove leading verse numbers (e.g., "1 The book..." -> "The book...")
    - Normalize whitespace
    - Remove extra spaces
    """
    # Remove leading verse numbers at the beginning of text
    text = re.sub(r'^\d+\s+', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

def is_valid_verse(text):
    """
    Check if a verse is valid (not empty, not marked as MISSING)
    """
    if not text or text.strip() == "":
        return False
    if "MISSING" in text:
        return False
    return True

## Clean and Process Data

Apply cleaning functions to all verses and organize by language.

In [7]:
# Clean and process all verses
cleaned_data = {}
stats = {lang: {"total": 0, "valid": 0, "invalid": 0} for lang in languages}

for lang in languages:
    cleaned_data[lang] = {}
    
    for key, text in raw_data[lang].items():
        cleaned_text = clean_text(text)
        
        stats[lang]["total"] += 1
        if is_valid_verse(cleaned_text):
            cleaned_data[lang][key] = cleaned_text
            stats[lang]["valid"] += 1
        else:
            stats[lang]["invalid"] += 1

# Display statistics
print("Cleaning Statistics:")
print("=" * 60)
for lang in languages:
    s = stats[lang]
    print(f"{lang:12} | Total: {s['total']:4} | Valid: {s['valid']:4} | Invalid: {s['invalid']:3}")
print("=" * 60)

Cleaning Statistics:
english      | Total: 3007 | Valid: 3007 | Invalid:   0
tagalog      | Total: 3033 | Valid: 3033 | Invalid:   0
bikolano     | Total: 3141 | Valid: 3141 | Invalid:   0
ilonggo      | Total: 3124 | Valid: 3124 | Invalid:   0
waray        | Total: 3134 | Valid: 3134 | Invalid:   0
masbatenyo   | Total: 3079 | Valid: 3079 | Invalid:   0


## Identify Common Verses Across Languages

In [8]:
# Find verses that exist in all languages
all_verse_keys = [set(cleaned_data[lang].keys()) for lang in languages]
common_verses = set.intersection(*all_verse_keys)

print(f"Total common verses across all {len(languages)} languages: {len(common_verses)}")

# Sort common verses by book, chapter, verse
common_verses_sorted = sorted(list(common_verses))

# Display breakdown by book
books_count = defaultdict(int)
for book, chapter, verse in common_verses_sorted:
    books_count[book] += 1

print("\nBreakdown by book:")
for book in sorted(books_count.keys()):
    print(f"  {book}: {books_count[book]} verses")

Total common verses across all 6 languages: 2948

Breakdown by book:
  LUK: 1171 verses
  MAT: 1090 verses
  MRK: 687 verses


## Create Segmented Data Structure

Organize cleaned verses by language with their references for easy parallel corpus creation.

In [9]:
# Create a structured dataset for each language
# Format: list of dicts with keys: book, chapter, verse, text, reference
segmented_data = {}

for lang in languages:
    segmented_data[lang] = []
    
    for key in common_verses_sorted:
        book, chapter, verse = key
        text = cleaned_data[lang][key]
        
        segmented_data[lang].append({
            "book": book,
            "chapter": chapter,
            "verse": verse,
            "reference": f"{book} {chapter}:{verse}",
            "text": text
        })

# Display sample from each language
print("Sample verses from each language:")
print("=" * 80)
for lang in languages:
    print(f"\n{lang.upper()}:")
    sample = segmented_data[lang][0]  # First verse
    print(f"  Reference: {sample['reference']}")
    print(f"  Text: {sample['text'][:100]}...")
print("=" * 80)

Sample verses from each language:

ENGLISH:
  Reference: LUK 1:1
  Text: Forasmuch as many have taken in hand to draw up a narrative concerning those matters which have been...

TAGALOG:
  Reference: LUK 1:1
  Text: Yamang marami ang nagsikap mag-ayos ng isang kasaysayan tungkol sa mga bagay na naganap sa gitna nat...

BIKOLANO:
  Reference: LUK 1:1
  Text: Ginagalangan kong Teofilo:...

ILONGGO:
  Reference: LUK 1:1
  Text: Halangdon nga Teofilo:...

WARAY:
  Reference: LUK 1:1
  Text: Hinigugma ko nga Teofilo:...

MASBATENYO:
  Reference: LUK 1:1
  Text: Ginagalangan na Teofilo, damo na an nagtalinguha pagtipon sin mga istorya manungod san mga nangyari ...


## Save Processed Data

Save the cleaned and segmented data in multiple formats for easy use in the next notebook.

In [10]:
# Create processed directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save each language as JSON (with metadata)
for lang in languages:
    output_file = processed_dir / f"{lang}_clean.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(segmented_data[lang], f, ensure_ascii=False, indent=2)
    print(f"Saved {lang} to {output_file}")

# Save each language as simple text file (one verse per line)
for lang in languages:
    output_file = processed_dir / f"{lang}_clean.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        for verse in segmented_data[lang]:
            f.write(f"{verse['text']}\n")
    print(f"Saved {lang} to {output_file}")

# Save common verse references
references_file = processed_dir / "verse_references.txt"
with open(references_file, "w", encoding="utf-8") as f:
    for book, chapter, verse in common_verses_sorted:
        f.write(f"{book} {chapter}:{verse}\n")
print(f"\nSaved verse references to {references_file}")

print(f"\n✓ All processed data saved to {processed_dir}")

Saved english to ..\data\processed\english_clean.json
Saved tagalog to ..\data\processed\tagalog_clean.json
Saved bikolano to ..\data\processed\bikolano_clean.json
Saved ilonggo to ..\data\processed\ilonggo_clean.json
Saved waray to ..\data\processed\waray_clean.json
Saved masbatenyo to ..\data\processed\masbatenyo_clean.json
Saved english to ..\data\processed\english_clean.txt
Saved tagalog to ..\data\processed\tagalog_clean.txt
Saved bikolano to ..\data\processed\bikolano_clean.txt
Saved ilonggo to ..\data\processed\ilonggo_clean.txt
Saved waray to ..\data\processed\waray_clean.txt
Saved masbatenyo to ..\data\processed\masbatenyo_clean.txt

Saved verse references to ..\data\processed\verse_references.txt

✓ All processed data saved to ..\data\processed
Saved masbatenyo to ..\data\processed\masbatenyo_clean.json
Saved english to ..\data\processed\english_clean.txt
Saved tagalog to ..\data\processed\tagalog_clean.txt
Saved bikolano to ..\data\processed\bikolano_clean.txt
Saved ilonggo 

## Summary Statistics

Display final statistics about the cleaned and processed data.

In [11]:
# Calculate statistics
print("\n" + "=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)

print(f"\nTotal languages: {len(languages)}")
print(f"Languages: {', '.join(languages)}")

print(f"\nCommon verses across all languages: {len(common_verses_sorted)}")

print("\nWord count statistics:")
for lang in languages:
    total_words = sum(len(verse['text'].split()) for verse in segmented_data[lang])
    avg_words = total_words / len(segmented_data[lang]) if segmented_data[lang] else 0
    print(f"  {lang:12}: {total_words:6,} words | Avg per verse: {avg_words:.1f}")

print("\nBooks included:")
for book in sorted(books_count.keys()):
    print(f"  {book}: {books_count[book]} verses")

print("\n" + "=" * 80)
print("✓ Data is ready for parallel corpus creation!")
print("=" * 80)


PREPROCESSING SUMMARY

Total languages: 6
Languages: english, tagalog, bikolano, ilonggo, waray, masbatenyo

Common verses across all languages: 2948

Word count statistics:
  english     : 62,810 words | Avg per verse: 21.3
  tagalog     : 62,619 words | Avg per verse: 21.2
  bikolano    : 57,918 words | Avg per verse: 19.6
  ilonggo     : 67,835 words | Avg per verse: 23.0
  waray       : 65,094 words | Avg per verse: 22.1
  masbatenyo  : 65,377 words | Avg per verse: 22.2

Books included:
  LUK: 1171 verses
  MAT: 1090 verses
  MRK: 687 verses

✓ Data is ready for parallel corpus creation!
